In [12]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import librosa
import itertools
import io
import numpy as np
import json

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [19]:
!ls ChildMandarin/new_data/dev

README.md  new_data  speaker_info.xlsx


In [15]:
import tarfile

for f in glob('ChildMandarin/new_data/*.tar'):

    with tarfile.open(f, "r") as tar:
        tar.extractall(path='ChildMandarin/new_data')

In [20]:
files = glob('ChildMandarin/new_data/*/**/*.json')
len(files)

40685

In [24]:
!mkdir ChildMandarin_audio

In [41]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files
    data = []
    for f in tqdm(files):
        try:
            with open(f) as fopen:
                d = json.load(fopen)
    
            t = d['text'].strip()
            if len(t) < 2:
                continue
    
            audio_np, sr = sf.read(f.replace('.json', '.wav'))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            audio_filename = os.path.join('ChildMandarin_audio', d['id'] + '.mp3')
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"ChildMandarin_{d['speaker_id']}"
            })
        except Exception as e:
            pass
    return data

In [42]:
data = loop((files[:10], 0))

100%|██████████| 10/10 [00:00<00:00, 40.15it/s]


In [43]:
len(data)

10

In [44]:
data[0]

{'audio_filename': 'ChildMandarin_audio/370_3_M_H_SHANXI_iPhone_001_062.mp3',
 'text': '老，老师。',
 'speaker': 'ChildMandarin_370'}

In [45]:
# import IPython.display as ipd
# ipd.Audio(data[0]['audio_filename'])

In [46]:
data = multiprocessing(files, loop, cores = min(10, len(files)))

100%|██████████| 4068/4068 [02:44<00:00, 24.68it/s]


In [47]:
len(data)

40637

In [48]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'ChildMandarin_audio/370_3_M_H_SHANXI_iPhone_001_062.mp3',
 'text': '老，老师。',
 'speaker': 'ChildMandarin_370'}

In [49]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'ChildMandarin')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 70.71ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 1.21MB / 1.21MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 1.21MB / 1.21MB,  0.00B/s  
New Data Upload: 100%|██████████| 1.21MB / 1.21MB,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.25 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/ff38f5346a8fbb92cfe606c91ab75bb4bb87285b', commit_message='Upload dataset', commit_description='', oid='ff38f5346a8fbb92cfe606c91ab75bb4bb87285b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [51]:
audio_files = [d['audio_filename'] for d in data]

with open('ChildMandarin-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [53]:
folders = glob('ChildMandarin_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

ChildMandarin_audio_neucodec
ChildMandarin_audio


In [54]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('ChildMandarin_audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  99%|█████████▉| 32.3MB / 32.7MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 32.7MB / 32.7MB,  754kB/s  
Processing Files (1 / 1): 100%|██████████| 32.7MB / 32.7MB,  599kB/s  
New Data Upload: 100%|██████████| 32.7MB / 32.7MB,  599kB/s  
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   6%|▌         | 45.6MB /  744MB,   ???B/s  
Processing Files (0 / 1):  19%|█▊        |  138MB /  744MB,  462MB/s  
Processing Files (0 / 1):  41%|████      |  304MB /  744MB,  647MB/s  
Processing Files (0 / 1):  46%|████▌     |  341MB /  744MB,  492MB/s  
Processing Files (0 / 1):  60%|██████    |  450MB /  744MB,  505MB/s  
Processing Files (0 / 1):  84%|████████▍ |  627MB /  744MB,  582MB/s  
Processing Files (0 / 1): 100%|█████████▉|  741MB /  744MB,  580MB/s  
Processing Files (0 / 1): 100%|█████████▉|  742MB /  744MB,  435MB/s  
Processing Files (0 / 1